# Phase 3 — Mediator Flow (Habermas Machine)

Runs the full Mediator approach on the same opinions used in Phase 1.

**What this does differently from Phases 1–2:**
Participants never vote. Instead, the LLM generates N candidate group statements
from the raw opinions, predicts how each participant would rank them, and Schulze
voting picks the winner — all without any additional input from the group.

**Why run both?**
The Mediator is faster and needs nothing beyond opinions. The atomic/voting approach
is slower but produces democratic legitimacy (people voted), reveals group structure
(clusters), and surfaces explicit disagreements. This notebook lets you compare the
two outputs side by side.

**Run Phases 1–2 first** if you want a comparison in Step 5.

In [ ]:
# ── CONFIG ───────────────────────────────────────────────────────────────────
TOPIC          = "Where should we hold the combined summer social for engineering and marketing?"
OPINIONS_CSV   = "example_opinions.csv"
NUM_CANDIDATES = 5   # candidate group statements to generate per round
SESSION_ID     = "session_01"
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import asyncio
import os
import sys
from pathlib import Path
import nest_asyncio

nest_asyncio.apply()

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv(PROJECT_ROOT / ".env")

import pandas as pd
from group_consensus.mediation.async_mediator import AsyncMediator
from group_consensus.mediation.social_choice import schulze_ranking
from group_consensus.models.types import Opinion, Participant, SessionConfig

OUTPUT_DIR = Path("session_data")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"API key : {'✓' if os.getenv('ANTHROPIC_API_KEY') else '⚠  not found'}")

---
## Step 1 — Load opinions

In [ ]:
df = pd.read_csv(OPINIONS_CSV)
print(f"{len(df)} responses — columns: {list(df.columns)}")

NAME_COL    = df.columns[1]
OPINION_COL = df.columns[2]

participants = []
opinions     = []
for i, row in df.iterrows():
    pid  = f"p{i}"
    name = str(row[NAME_COL]).strip()
    text = str(row[OPINION_COL]).strip()
    participants.append(Participant(id=pid, name=name))
    opinions.append(Opinion(participant_id=pid, text=text, session_id=SESSION_ID))

name_to_pid = {p.name: p.id for p in participants}
print(f"✓ {len(participants)} participants loaded")

---
## Step 2 — Run the Mediator

The LLM generates `NUM_CANDIDATES` candidate group statements, then predicts how each
participant would rank them based on their opinion. Schulze voting picks the winner.

No `critique_fn` is passed, so the loop stops after one round.

In [ ]:
config   = SessionConfig(
    session_id=SESSION_ID,
    topic=TOPIC,
    num_candidate_statements=NUM_CANDIDATES,
)
mediator = AsyncMediator(config)

result = asyncio.run(
    mediator.run(topic=TOPIC, participants=participants, opinions=opinions)
)

---
## Step 3 — All candidates and Schulze ranking

Shows every candidate statement generated in round 1, ranked by the Schulze method
from most to least preferred across all participants.

In [ ]:
round0      = result.rounds[0]
candidates  = round0.candidate_statements
rankings    = round0.rankings

# Reconstruct Schulze ranking over candidate indices
id_to_idx = {s.id: i for i, s in enumerate(candidates)}
index_rankings = [
    [id_to_idx[sid] for sid in r.statement_ids if sid in id_to_idx]
    for r in rankings
]
ranked_indices = schulze_ranking(index_rankings, len(candidates))

DIVIDER = "═" * 60
print(DIVIDER)
print("  CANDIDATE STATEMENTS — Schulze rank order (1 = winner)")
print(DIVIDER)
for rank, idx in enumerate(ranked_indices, 1):
    marker = "  ← Schulze winner" if rank == 1 else ""
    print(f"\n  {rank}. {candidates[idx].text}{marker}")
print()

---
## Step 4 — Final statement

In [ ]:
DIVIDER = "═" * 60
print(DIVIDER)
print("  MEDIATOR CONSENSUS STATEMENT")
print(DIVIDER)
print()
print(result.consensus_statement.text)
print()

out_path = OUTPUT_DIR / "mediator_statement.txt"
with open(out_path, "w") as f:
    f.write(result.consensus_statement.text)
print(f"✓ Saved to {out_path}")

---
## ⏸  Optional — Add critiques for a second round

Review the candidates above. If the winner misses something important — a constraint
that didn't surface, a tension that was papered over — add critiques below using
participant names as they appear in the CSV.

The mediator will use these to generate a refined set of candidates in round 2,
then Schulze-vote on those. Leave `CRITIQUES` empty to skip.

In [ ]:
# ── ENTER CRITIQUES HERE ──────────────────────────────────────────────────────
# Key = participant name exactly as in the CSV. Value = their critique.
# Uncomment any that apply after reviewing the round 1 winner above.
CRITIQUES = {
    # "Sarah Chen":       "No mention of an end time — I need to leave by 8pm, this is a hard constraint not a preference",
    # "James Okafor":     "The statement says nothing about cross-team mixing or a shared activity; that's the whole reason we're doing this",
    # "Priya Sharma":     "Step-free access is mentioned but needs to say 'throughout the venue', not just at the entrance",
    # "Dan Mitchell":     "This reads like a requirements checklist, not an evening out — it should say something about a relaxed, informal atmosphere",
    # "Emma Roberts":     "Private space is good but there's nothing about how people who don't know each other will actually connect",
    # "Kwame Asante":     "Still no mention of a structured icebreaker — without one people will just cluster with their own team again",
    # "Lisa Park":        "Should be explicit that company covers food AND drinks, not just one or the other",
    # "Tom Walsh":        "The statement is all logistics, nothing about it feeling relaxed and voluntary rather than organised and managed",
    # "Aisha Johnson":    "Good on practical points but doesn't capture the atmosphere — it should feel like a genuine celebration, not a facilities audit",
    # "Marcus Lee":       "Needs to say something about energy and atmosphere — a venue that ticks all these boxes but feels dead is no good",
    # "Yuki Tanaka":      "Step-free access is there but I need the format itself to avoid prolonged standing — seating throughout matters",
    # "Ben Clarke":       "Nothing about a set menu, which is the only sensible format for a group this size",
    # "Fatima Al-Hassan": "Non-alcoholic options are mentioned but 'strong' is vague — I need proper options, not just tap water",
    # "Ryan O'Brien":     "Pure logistics, nothing about giving people something to do together — an evening that's just dinner feels flat",
    # "Nina Petrov":      "No acknowledgement that the whole point is the two teams actually connecting, not just being in the same room",
}
# ─────────────────────────────────────────────────────────────────────────────

unknown = [name for name in CRITIQUES if name not in name_to_pid]
if unknown:
    print(f"⚠  Names not found in participants: {unknown}")
else:
    print(f"{len(CRITIQUES)} critique(s) entered — run the next cell to refine.")

In [ ]:
if not CRITIQUES:
    print("No critiques entered — skipping second round.")
else:
    critique_opinions = [
        Opinion(participant_id=name_to_pid[name], text=text, session_id=SESSION_ID)
        for name, text in CRITIQUES.items()
        if name in name_to_pid
    ]

    # Return critiques on first call only — stops after one refinement round
    _critique_used = [False]
    def critique_fn(winner, participants):
        if _critique_used[0]:
            return []
        _critique_used[0] = True
        return critique_opinions

    config2  = SessionConfig(
        session_id=SESSION_ID,
        topic=TOPIC,
        num_candidate_statements=NUM_CANDIDATES,
        max_deliberation_rounds=2,
    )
    mediator2 = AsyncMediator(config2)
    result2   = asyncio.run(
        mediator2.run(topic=TOPIC, participants=participants, opinions=opinions,
                      critique_fn=critique_fn)
    )

    DIVIDER = "═" * 60
    print(DIVIDER)
    print("  REFINED STATEMENT (round 2)")
    print(DIVIDER)
    print()
    print(result2.consensus_statement.text)

    refined_path = OUTPUT_DIR / "mediator_statement_refined.txt"
    with open(refined_path, "w") as f:
        f.write(result2.consensus_statement.text)
    print(f"\n✓ Saved to {refined_path}")

    result = result2  # carry refined result into comparison below

---
## Step 5 — Compare with the atomic/voting approach

What to look for:

**Mediator statement** — generated holistically from raw opinions. Tends toward
smoother prose that acknowledges tensions, but the Schulze ranking is an AI prediction
of preference, not an actual vote. Participants were never shown the statements.

**Atomic/voting statement** — built from the specific points the group actually agreed
on. Structurally grounded in real votes. Also surfaces group clusters and divisive
statements the Mediator misses entirely.

Neither is strictly better. The Mediator is faster; the voting approach has democratic
legitimacy and reveals group structure.

In [ ]:
DIVIDER = "═" * 60

print(DIVIDER)
print("  APPROACH A — Mediator (this notebook)")
print("  Source: AI-predicted Schulze winner from raw opinions")
print(DIVIDER)
print()
print(result.consensus_statement.text)
print()

atomic_path = OUTPUT_DIR / "group_statement.txt"
if atomic_path.exists():
    print(DIVIDER)
    print("  APPROACH B — Atomic + voting (01_mediate.ipynb + 02_analyse.ipynb)")
    print("  Source: synthesised from statements the group actually voted to agree on")
    print(DIVIDER)
    print()
    print(atomic_path.read_text().strip())
    print()
else:
    print("[Run 01_mediate.ipynb and 02_analyse.ipynb first to see the atomic/voting output here.]")